In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [2]:
# dataset generation
import pandas as pd
import numpy as np
import category_encoders as ce

# Simulating a dataset
data = {
    'Age': np.random.randint(20, 60, size=100).astype(float),  # Random ages between 20 and 60
    'State': np.random.choice(['Karnataka', 'Tamil Nadu', 'Maharashtra', 'Delhi', 'Telangana'], size=100),
    'Education': np.random.choice(['High School', 'UG', 'PG'], size=100),
    'Package': np.random.rand(100) * 100  # Random package values for demonstration
}

# Introducing missing values in 'Age' column (5%)
np.random.seed(0)  # For reproducibility
missing_indices = np.random.choice(data['Age'].shape[0], replace=False, size=int(data['Age'].shape[0] * 0.05))
data['Age'][missing_indices] = np.nan

df = pd.DataFrame(data)

df.head()

,Age,State,Education,Package
0,34.0,Telangana,PG,76.745417
1,36.0,Maharashtra,High School,9.686825
2,NaN,Maharashtra,PG,28.264267
3,59.0,Karnataka,PG,49.892482
4,30.0,Delhi,UG,66.037696


In [3]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df.drop(columns=['Package']),df['Package'],test_size=0.2,random_state=42)


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator ,TransformerMixin

In [5]:
from sklearn.base import BaseEstimator, TransformerMixin

class CountEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None):
        # columns: list of categorical columns to encode
        self.columns = columns
        # dictionary to store value counts for each column
        self.count_map = {}

    def fit(self, X, y=None):
        # If columns are not specified, use all columns
        if self.columns is None:
            self.columns = X.columns

        # Create count mapping for each column
        for col in self.columns:
            self.count_map[col] = X[col].value_counts().to_dict()

        return self  # Required for sklearn compatibility

    def transform(self, X):
        X = X.copy()

        # Replace categories with their counts
        for col in self.columns:
            X[col] = X[col].map(self.count_map[col]).fillna(0)

        return X


In [6]:
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('age_missing', SimpleImputer(strategy='mean'), ['Age']),
        ('cat_state', CountEncoder(), ['State']),
        ('education_ordinal', OrdinalEncoder(), ['Education'])
    ]
)

sklearn.set_config(transform_output='pandas')


In [7]:
preprocessor.fit_transform(X_train)

,age_missing__Age,cat_state__State,education_ordinal__Education
55,41.453333,19,2.0
88,53.000000,18,0.0
26,41.453333,18,2.0
42,38.000000,19,2.0
69,38.000000,12,2.0
...,...,...,...
60,52.000000,18,1.0
71,28.000000,18,0.0
14,43.000000,20,0.0
92,57.000000,11,0.0


In [8]:
X_train

,Age,State,Education
55,NaN,Maharashtra,UG
88,53.0,Tamil Nadu,High School
26,NaN,Tamil Nadu,UG
42,38.0,Maharashtra,UG
69,38.0,Telangana,UG
...,...,...,...
60,52.0,Tamil Nadu,PG
71,28.0,Tamil Nadu,High School
14,43.0,Karnataka,High School
92,57.0,Delhi,High School


In [9]:
df['State'].value_counts()

State
Maharashtra    22
Karnataka      22
Tamil Nadu     21
Telangana      20
Delhi          15
Name: count, dtype: int64